⏱️ **Time required:** ~2 minutes | **Type:** Domain pipeline (run all cells)

# 💳 RideFlow Payments — Domain Pipeline

**Domain Owner:** Finance Engineering  
**System:** stripe  

This notebook represents the **Payments domain team's** autonomous pipeline. It generates synthetic Stripe transaction data (charges, payouts, refunds), processes it through the Medallion architecture, and produces a **Payment Reconciliation** Gold table that cross-references trip data from the Marketplace domain.

| Published Data Product | Layer | Description |
| :-- | :-- | :-- |
| `silver_stripe_charges` | Silver | Typed, deduplicated charge records |
| `silver_stripe_refunds` | Silver | Cleaned refund records |
| `gold_stripe_payment_reconciliation` | Gold | Cross-domain trip ↔ payment matching |

> **Cross-Domain Dependency:** This pipeline reads `silver_rideflow_trips` from the Marketplace domain to reconcile `trip_id → charge_id`. Run `07_rideflow_marketplace.ipynb` first.

---
## Step 1 · Environment Setup

In [ ]:
import os, sys
from pathlib import Path
import polars as pl
import lakelogic as ll

PROJECT_ROOT = Path(".").resolve()
LAKEHOUSE = PROJECT_ROOT / "lakehouse"
ENV = "local"

print(f"Project Root : {PROJECT_ROOT}")
print(f"Lakehouse    : {LAKEHOUSE}")

---
## Step 2 · Load Payments Registry

In [ ]:
from lakelogic.core.registry import DomainRegistry

registry = DomainRegistry.from_yaml(
    str(PROJECT_ROOT / "assets" / "domains_rideflow" / "payments" / "stripe" / "_system.yaml")
)

print(f"Domain  : {registry.domain}")
print(f"System  : {registry.system}")
print(f"Entities: {len(registry.get_active_contracts())} active contracts")

for c in registry.get_active_contracts():
    print(f"  [{c.target_layer:6s}] {c.entity}")

---
## Step 3 · Generate Synthetic Stripe Data

We generate realistic payment data that references trip IDs from the Marketplace domain, creating authentic cross-domain foreign key relationships.

In [ ]:
import random
import string
from datetime import datetime, timezone, timedelta

# Read Marketplace trip IDs to create realistic FK references
marketplace_trips_path = LAKEHOUSE / "marketplace" / "silver" / "silver_rideflow_trips"

if marketplace_trips_path.exists():
    trips_df = pl.read_delta(str(marketplace_trips_path))
    trip_ids = trips_df.select("trip_id").unique().to_series().to_list()
    rider_ids = trips_df.select("rider_id").unique().to_series().to_list()
    fare_amounts = trips_df.select("fare_amount").to_series().cast(pl.Float64).to_list()
    print(f"✅ Loaded {len(trip_ids)} trip IDs from Marketplace domain")
else:
    print("⚠️ Marketplace Silver trips not found. Run 07_rideflow_marketplace.ipynb first.")
    trip_ids = [f"TRP-{i:06d}" for i in range(500)]
    rider_ids = [f"RDR-{i:06d}" for i in range(200)]
    fare_amounts = [round(random.uniform(5, 80), 2) for _ in range(500)]

In [ ]:
def gen_id(prefix, length=8):
    return f"{prefix}_{''.join(random.choices(string.ascii_lowercase + string.digits, k=length))}"

now = datetime.now(timezone.utc)
n_charges = min(len(trip_ids), 1500)

# ── Charges ──────────────────────────────────────────────────────────────
charges = []
for i in range(n_charges):
    trip_id = trip_ids[i % len(trip_ids)]
    fare = fare_amounts[i % len(fare_amounts)]
    charges.append({
        "charge_id": gen_id("ch"),
        "trip_id": trip_id,
        "rider_id": rider_ids[i % len(rider_ids)],
        "amount": round(fare * 100),  # Stripe uses cents
        "currency": random.choice(["usd", "gbp", "eur"]),
        "status": random.choices(["succeeded", "failed", "pending"], weights=[92, 5, 3])[0],
        "payment_method": random.choice(["card", "apple_pay", "google_pay"]),
        "created_at": (now - timedelta(hours=random.randint(1, 72))).isoformat(),
    })

# ── Refunds ────────────────────────────────────────────────────────────
refund_indices = random.sample(range(n_charges), min(80, n_charges))
refunds = []
for idx in refund_indices:
    ch = charges[idx]
    refunds.append({
        "refund_id": gen_id("re"),
        "charge_id": ch["charge_id"],
        "amount": ch["amount"] if random.random() > 0.3 else round(ch["amount"] * random.uniform(0.2, 0.8)),
        "currency": ch["currency"],
        "reason": random.choice(["duplicate", "fraudulent", "requested_by_customer"]),
        "status": "succeeded",
        "created_at": (now - timedelta(hours=random.randint(1, 48))).isoformat(),
    })

# ── Payouts ────────────────────────────────────────────────────────────
payouts = []
for _ in range(30):
    payouts.append({
        "payout_id": gen_id("po"),
        "amount": random.randint(50000, 500000),
        "currency": random.choice(["usd", "gbp", "eur"]),
        "status": random.choices(["paid", "pending", "in_transit"], weights=[80, 10, 10])[0],
        "arrival_date": (now - timedelta(days=random.randint(0, 14))).strftime("%Y-%m-%d"),
        "created_at": (now - timedelta(days=random.randint(1, 15))).isoformat(),
    })

df_charges = pl.DataFrame(charges)
df_refunds = pl.DataFrame(refunds)
df_payouts = pl.DataFrame(payouts)

print(f"Generated:")
print(f"  💳 {len(charges):,} charges")
print(f"  ↩️  {len(refunds):,} refunds")
print(f"  💰 {len(payouts):,} payouts")

---
## Step 4 · Land Synthetic Data

Write the generated data as CSV files into the landing zone, just like a real Stripe webhook integration would.

In [ ]:
landing_root = LAKEHOUSE / "_data" / "landing_payments" / "stripe"

for name, df in [("charges", df_charges), ("refunds", df_refunds), ("payouts", df_payouts)]:
    dest = landing_root / name
    dest.mkdir(parents=True, exist_ok=True)
    csv_path = dest / f"{name}.csv"
    df.write_csv(str(csv_path))
    print(f"  ✅ {name}: {len(df):,} rows → {csv_path}")

---
## Step 5 · Bronze Ingestion

In [ ]:
from lakelogic.pipeline.runner import LakehousePipeline

runner = LakehousePipeline(
    registry,
    engine="polars",
    storage_root=str(LAKEHOUSE)
)

print("Starting Bronze Ingestion for Payments domain...\n")

summary = runner.run(
    target_layers="bronze",
    dry_run=False,
    environment=ENV
)

print(summary)

---
## Step 6 · Silver Processing

In [ ]:
print("Starting Silver Processing for Payments domain...\n")

summary = runner.run(
    target_layers="silver",
    dry_run=False,
    environment=ENV
)

print(summary)

---
## Step 7 · Gold Processing (Payment Reconciliation)

In [ ]:
print("Starting Gold Processing for Payments domain...\n")

summary = runner.run(
    target_layers="gold",
    dry_run=False,
    environment=ENV
)

print(summary)

---
## Step 8 · Cross-Domain Join: Trips × Charges

This is the key Data Mesh moment: the Payments domain reads a **published data product** from the Marketplace domain to reconcile trip revenue with Stripe charges. Note: we read from Silver (the published product), never from Marketplace's internal Bronze tables.

In [ ]:
# Read Marketplace data product
marketplace_trips = pl.read_delta(str(LAKEHOUSE / "marketplace" / "silver" / "silver_rideflow_trips"))

# Read our own Silver charges
charges_path = LAKEHOUSE / "payments" / "silver" / "silver_stripe_charges"
if charges_path.exists():
    our_charges = pl.read_delta(str(charges_path))
else:
    our_charges = df_charges  # Fallback to in-memory

# Cross-domain join
reconciled = our_charges.join(
    marketplace_trips.select(["trip_id", "fare_amount", "city_code", "trip_type"]),
    on="trip_id",
    how="inner"
)

print(f"🔗 Cross-Domain Reconciliation")
print(f"   Marketplace trips : {len(marketplace_trips):,}")
print(f"   Stripe charges    : {len(our_charges):,}")
print(f"   Matched records   : {len(reconciled):,}")
print(f"   Match rate        : {len(reconciled)/len(our_charges)*100:.1f}%")

display(reconciled.head(10))

---
## ✅ Data Products Published

The Payments domain pipeline has completed. Published data products:

| Data Product | Consumers |
| :-- | :-- |
| `silver_stripe_charges` | Finance reporting, Fraud detection |
| `silver_stripe_refunds` | Customer support, Revenue leakage |
| `gold_stripe_payment_reconciliation` | Executive dashboard, Auditing |

### Cross-Domain Dependencies

```
Marketplace Domain                    Payments Domain
┌─────────────────────┐              ┌─────────────────────────┐
│ silver_rideflow_trips│─── trip_id ──▶│ silver_stripe_charges   │
│    (data product)   │              │ gold_payment_reconcil.  │
└─────────────────────┘              └─────────────────────────┘
```

### Next Notebooks

- **`09_rideflow_operations.ipynb`** — Operations domain (Zendesk, Checkr, Twilio)
- **`12_compliance_gdpr_rtbf.ipynb`** — Cross-domain privacy erasure
- **`14_data_mesh_dashboards.ipynb`** — Unified mesh observability